In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/telecom_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)
df['CLV'] = df['MonthlyCharges'] * df['tenure']


In [ ]:
# Q21: Churn prediction feature dataset using Pandas feature engineering
# feature engineering = creating or transforming columns so a machine learning model
# can learn from them (ML models need numbers, not text like 'Yes'/'No')
features = pd.DataFrame()

# keep identifying and continuous columns as-is
features['customerID'] = df['customerID']
features['tenure'] = df['tenure']
features['monthly_charges'] = df['MonthlyCharges']
features['total_charges'] = df['TotalCharges']
features['clv'] = df['CLV']

# convert Yes/No text columns to 1/0 integers using .astype(int)
# (df['Partner'] == 'Yes') produces True/False → .astype(int) makes it 1/0
features['is_senior'] = df['SeniorCitizen']
features['has_partner'] = (df['Partner'] == 'Yes').astype(int)
features['has_dependents']  = (df['Dependents'] == 'Yes').astype(int)
features['has_phone'] = (df['PhoneService'] == 'Yes').astype(int)
features['has_security'] = (df['OnlineSecurity'] == 'Yes').astype(int)
features['has_techsupport'] = (df['TechSupport'] == 'Yes').astype(int)
features['is_paperless'] = (df['PaperlessBilling'] == 'Yes').astype(int)

# one-hot encode contract type: each contract option becomes its own 0/1 column
# Two year contract = both columns 0 (it's implied by elimination)
features['contract_monthly'] = (df['Contract'] == 'Month-to-month').astype(int)
features['contract_yearly']  = (df['Contract'] == 'One year').astype(int)

# rule-based risk score: add points for each known churn risk factor
# apply() runs the lambda once per row; axis=1 means row-by-row (not column-by-column)
# df.loc[row.name, col] looks up the original df because some columns aren't in features
features['risk_score'] = features.apply(lambda row:
    # contract type: month-to-month = highest risk (+3), one year = medium (+1), two year = safe (+0)
    (3 if df.loc[row.name, 'Contract'] == 'Month-to-month' else 1 if df.loc[row.name, 'Contract'] == 'One year' else 0) +
    # tenure: newest customers leave most (+3), loyalty reduces risk over time
    (3 if row['tenure'] <= 12 else 2 if row['tenure'] <= 24 else 1 if row['tenure'] <= 48 else 0) +
    # no tech support = more likely to leave when issues arise (+2)
    (2 if df.loc[row.name, 'TechSupport'] == 'No' else 0) +
    # no online security = same reasoning (+2)
    (2 if df.loc[row.name, 'OnlineSecurity'] == 'No' else 0), axis=1)
# max possible score = 3 + 3 + 2 + 2 = 10 → confirmed by Q10 output showing score 10 = 64% churn

# convert the Churn label to 0/1 — this is the target variable any ML model would predict
features['churn'] = (df['Churn'] == 'Yes').astype(int)

features.head()

In [ ]:
# Q23: Retention dashboard dataset with retention KPIs (Optional)

In [4]:
# Q24: Rule-based churn detection engine
def churn_risk_level(row):
    score = 0

    if row['Contract'] == 'Month-to-month':
        score += 3
    elif row['Contract'] == 'One year':
        score += 1

    if row['tenure'] <= 12:
        score += 3
    elif row['tenure'] <= 24:
        score += 2
    elif row['tenure'] <= 48:
        score += 1

    if row['OnlineSecurity'] == 'No':
        score += 2
    if row['TechSupport'] == 'No':
        score += 2

    if row['PaymentMethod'] == 'Electronic check':
        score += 1

    if row['SeniorCitizen'] == 1:
        score += 1

    if score >= 9:
        return 'Critical'
    elif score >= 6:
        return 'High'
    elif score >= 3:
        return 'Medium'
    else:
        return 'Low'

df['churn_risk'] = df.apply(churn_risk_level, axis=1)

df.groupby('churn_risk').agg(
    customer_count = ('customerID', 'count'),
    churn_rate     = ('Churn', lambda x: round((x == 'Yes').sum() / len(x) * 100, 2))
).reset_index().sort_values('churn_rate', ascending=False)

,churn_risk,customer_count,churn_rate
0,Critical,2105,56.72
1,High,1764,27.61
3,Medium,1377,9.37
2,Low,1797,3.28
